In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer - פונקציית Qiskit מבית Kipu Quantum

> **Note:** * Qiskit Functions הן תכונה ניסיונית הזמינה רק למשתמשי תוכנית IBM Quantum&reg; Premium Plan, Flex Plan, ו-On-Prem (דרך IBM Quantum Platform API). הן בסטטוס גרסת תצוגה מקדימה וכפופות לשינויים.

## סקירה כללית
עם Iskay Quantum Optimizer מבית Kipu Quantum, אפשר להתמודד עם בעיות אופטימיזציה מורכבות באמצעות מחשבים קוונטיים של IBM&reg;. הפותר הזה מנצל את האלגוריתם החדשני [bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) של Kipu, שדורש רק את פונקציית המטרה כקלט ומספק אוטומטית פתרונות לבעיה. הוא מסוגל לטפל בבעיות אופטימיזציה הכוללות עד 156 Qubit, מה שמאפשר שימוש בכל ה-Qubit של מכשירי הקוונטום של IBM. ה-Optimizer משתמש במיפוי 1-ל-1 בין משתנים קלאסיים ל-Qubit, מה שמאפשר לך להתמודד עם בעיות אופטימיזציה עם עד 156 משתנים בינאריים.

ה-Optimizer מאפשר פתרון בעיות אופטימיזציה בינארית ללא אילוצים. בנוסף לפורמולציה הנפוצה QUBO (Quadratic Unconstrained Binary Optimization), הוא תומך גם בבעיות אופטימיזציה מסדר גבוה יותר (HUBO). הפותר מנצל אלגוריתם קוונטי לא-וריאציוני, שמבצע את רוב החישוב על מכשירים קוונטיים.

הנ"ל מספק פרטים נוספים על האלגוריתם בו נעשה שימוש ומדריך קצר כיצד להשתמש בפונקציה, כמו גם תוצאות ביצועים השוואתיות על מגוון מופעי בעיות בגדלים ומורכבויות שונות.

## תיאור
ה-Optimizer הוא מימוש מוכן לשימוש של אלגוריתמי אופטימיזציה קוונטית חדשניים. הוא פותר בעיות אופטימיזציה על ידי הרצת Circuit קוונטיים דחוסים מאוד על חומרה קוונטית. דחיסה זו מושגת על ידי הכנסת מושגים counterdiabatic לאבולוציה בזמן של המערכת הקוונטית. האלגוריתם מבצע מספר איטרציות של ריצות חומרה כדי לקבל את הפתרונות הסופיים ומשלב אותן עם עיבוד לאחר. שלבים אלה משולבים בצורה חלקה בתהליך העבודה של ה-Optimizer ומבוצעים אוטומטית.

### איך עובד ה-Quantum Optimizer?
סעיף זה מתאר את הבסיס של אלגוריתם bf-DCQO המיושם. מבוא לאלגוריתם ניתן למצוא גם ב[ערוץ YouTube של Qiskit.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

האלגוריתם מבוסס על האבולוציה בזמן של מערכת קוונטית שמשתנה לאורך זמן, כאשר פתרון הבעיה מקודד במצב הבסיסי של המערכת הקוונטית בסוף האבולוציה. לפי [משפט האדיאבטי](https://en.wikipedia.org/wiki/Adiabatic_theorem), אבולוציה זו חייבת להיות איטית כדי להבטיח שהמערכת נשארת במצב הבסיסי שלה. דיגיטציה של אבולוציה זו היא הבסיס של חישוב קוונטי אדיאבטי דיגיטלי (DQA) ואלגוריתם QAOA המפורסם. עם זאת, האבולוציה האיטית הנדרשת אינה ישימה עבור גדלי בעיות הולכים וגדלים מכיוון שהיא מביאה לעומק Circuit הולך וגדל. על ידי שימוש בפרוטוקולים counterdiabatic, אפשר לדכא עירורים לא רצויים המתרחשים במהלך זמני אבולוציה קצרים תוך כדי שמירה על המצב הבסיסי. כאן, דיגיטציה של זמן אבולוציה קצר יותר זה מביאה ל-Circuit קוונטיים בעומק קצר יותר ועם פחות שערי קשירה.

ה-Circuit של אלגוריתמי bf-DCQO משתמשים בדרך כלל בעד עשר פעמים פחות שערי קשירה מאשר DQA, ובשלוש עד ארבע פעמים פחות שערי קשירה מאשר מימושי QAOA סטנדרטיים. בגלל מספר שערים קטן יותר, מתרחשות פחות שגיאות במהלך ביצוע ה-Circuit על חומרה. לכן, ה-Optimizer לא דורש שימוש בטכניקות כמו דיכוי שגיאות או הקלה על שגיאות. יישום שלהן בגרסאות עתידיות יכול לשפר עוד יותר את איכות הפתרון.

למרות שאלגוריתם bf-DCQO משתמש באיטרציות, הוא לא-וריאציוני. לאחר כל איטרציה של האלגוריתם, מחלוקת המצבים נמדדת. ההתפלגות שהתקבלה משמשת לחישוב מה שנקרא "שדה הטיה". שדה ההטיה מאפשר להתחיל את האיטרציה הבאה ממצב אנרגיה קרוב לפתרון שנמצא קודם. בדרך זו, האלגוריתם עובר עם כל איטרציה לפתרונות של אנרגיה נמוכה יותר. בדרך כלל, כ-עשר איטרציות מספיקות להתכנסות לפתרון, בסה"כ דורש מספר נמוך בהרבה של איטרציות מאשר אלגוריתמים וריאציוניים, שהוא בסדר גודל של כ-100 איטרציות.

ה-Optimizer משלב את אלגוריתם bf-DCQO עם עיבוד לאחר קלאסי. לאחר מדידת התפלגות המצבים, מתבצע חיפוש מקומי. במהלך החיפוש המקומי, הסיביות של הפתרון הנמדד מתהפכות באופן אקראי. לאחר ההיפוך, האנרגיה של המחרוזת הבינארית החדשה מוערכת. אם האנרגיה נמוכה יותר, המחרוזת הבינארית נשמרת כפתרון החדש. החיפוש המקומי מתרחב רק ליניארית עם מספר ה-Qubit; לכן, הוא זול מבחינה חישובית. מכיוון שהעיבוד לאחר מתקן היפוכי סיביות מקומיים, הוא מפצה על שגיאות היפוך סיבית שלעיתים קרובות הן תוצאה של פגמי חומרה ושגיאות קריאה.

### תהליך עבודה
להלן סכמה של תהליך העבודה של ה-Quantum Optimizer.

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Workflow of the Quantum Optimizer")

על ידי שימוש ב-Quantum Optimizer, פתרון בעיית אופטימיזציה על חומרה קוונטית ניתן לצמצם ל:
* ניסוח פונקציית המטרה של הבעיה
* גישה ל-Optimizer דרך Qiskit Functions
* הרצת ה-Optimizer ואיסוף התוצאה

## ביצועים השוואתיים
מדדי הביצועים ההשוואתיים שלהלן מראים שה-Optimizer מטפל ביעילות בבעיות הכוללות עד 156 Qubit ומציעים סקירה כללית של דיוק ה-Optimizer ויכולת ההרחבה שלו על פני סוגי בעיות שונים. שים לב שמדדי הביצועים בפועל עשויים להשתנות בהתאם למאפייני הבעיה הספציפיים, כגון מספר המשתנים, הצפיפות והמקומיות של המונחים בפונקציית המטרה, והסדר הפולינומי.

הטבלה הבאה כוללת את יחס הקירוב (AR), מדד המוגדר כדלקמן:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
כאשר $C$ היא פונקציית המטרה, $C_{\textrm{min}}$, $C_{\textrm{max}}$ הם הערכים המינימלי והמקסימלי שלה, ו-$C^{*}$ הוא העלות של הפתרון הטוב ביותר שנמצא, בהתאמה. לכן, AR=100\% אומר שהמצב הבסיסי של הבעיה הושג.

| דוגמה           | מספר Qubit | יחס קירוב | זמן כולל (שניות) | שימוש בזמן ריצה (שניות) | מספר כולל של shots | מספר איטרציות |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- מופעי MaxCut עם 28, 30 ו-32 Qubit הורצו על ibm_sherbrooke. מופעים עם 80, 100 ו-120 הורצו על מעבד Heron r2.
- מופעי HUBO גם הם הורצו על מעבד Heron r2.

כל מופעי הביצועים ההשוואתיים נגישים ב-GitHub (ראה [מופעי ביצועים של Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). דוגמה להרצת מופעים אלה ניתן למצוא ב[דוגמה 3: מופעי ביצועים](#example-3-benchmark-instances).

## קלט ופלט
### קלט
ראה את הטבלה הבאה לכל פרמטרי הקלט ש-Quantum Optimizer מקבל. הסעיף **Options** שלאחר מכן נכנס לפרטים נוספים לגבי ה-`options` הזמינות.

| שם         | סוג               | תיאור                                                                                                                                                                            | נדרש | ברירת מחדל                                                                                       | דוגמה                                                                                                              |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | המקדמים של בעיית האופטימיזציה המנוסחת כ-QUBO/HUBO או בפורמט spin. למידע נוסף על מפרט הבעיה, ראה [פורמטי בעיה מקובלים](#accepted-problem-formats)                                       | כן      | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | ציין האם מקדמי הבעיה הם בפורמט בינארי (QUBO/HUBO) או spin. שתי האפשרויות הן `"spin"` או `"binary"`                                                   | כן      | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | שם ה-Backend לביצוע הבקשה                                                                                                                                                  | כן       | N/A                                                      | `"ibm_fez"`                                                                                                          |
| options      | `Dict[str, Any]`   | אפשרויות לטיפול בהגשת החומרה, כגון מספר ה-shots. לפרטים נוספים על תצורת האפשרויות, ראה את הסעיף [Options](#options) | לא       | לצפייה בערכי ברירת המחדל של תצורת האפשרויות ראה את הסעיף [Options](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |


הארגומנטים `problem` ו-`problem_type` מקודדים בעיית אופטימיזציה בצורה

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
כאשר

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- על ידי בחירת `problem_type = "binary"`, אתה מציין שפונקציית העלות היא בפורמט `binary`, כלומר ש-$D = {0,  1}^{n}$, כלומר, פונקציית העלות כתובה בפורמולציית QUBO/HUBO.
- מצד שני, על ידי בחירת `problem_type = "spin"`, פונקציית העלות כתובה בפורמולציית Ising, כאשר $D = {-1, 1}^{n}$.

יש לקדד את מקדמי הבעיה במילון כדלקמן:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- שים לב שמפתחות המילון חייבים להיות מחרוזות המכילות tuple חוקי של מספרים שלמים שאינם חוזרים על עצמם.

#### Options
Iskay מספקת יכולות כוונון עדין דרך פרמטרים אופציונליים. בעוד שברירות המחדל עובדות היטב לרוב הבעיות, אפשר להתאים אישית את ההתנהגות לדרישות ספציפיות:

| פרמטר | סוג | ברירת מחדל | תיאור |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | מדידות קוונטיות לכל איטרציה (יותר = דיוק גבוה יותר) |
| **num_iterations** | `int` | 10 | איטרציות של האלגוריתם (יותר איטרציות יכולות לשפר את איכות הפתרון) |
| **use_session** | `bool` | True | שימוש ב-Session של IBM לזמני תור מופחתים |
| **seed_transpiler** | `int` | None | הגדר לקבלת קומפילציה רבייתית של Circuit קוונטי |
| **direct_qubit_mapping** | `bool` | False | מיפוי Qubit וירטואלי ישירות ל-Qubit פיזי |
| **job_tags** | `List[str]` | None | תגיות מותאמות אישית למעקב אחר עבודות |
| **preprocessing_level** | `int` | 0 | עוצמת עיבוד מקדים של הבעיה (0-3) - ראה פרטים להלן |
| **postprocessing_level** | `int` | 2 | רמת עידון הפתרון (0-2) - ראה פרטים להלן |
| **transpilation_level** | `int` | 0 | ניסיונות אופטימיזציה של ה-Transpiler (0-5) - ראה פרטים להלן |
| **transpile_only** | `bool` | False | ניתוח אופטימיזציה של Circuit ללא הרצת ביצוע מלאה |

**רמות עיבוד מקדים (0-3)**: חשובות במיוחד לבעיות גדולות יותר שאינן יכולות כרגע להתאים לזמני הקוהרנטיות של החומרה. רמות עיבוד מקדים גבוהות יותר משיגות עומקי Circuit רדודים יותר על ידי קירובים בתרגום הבעיה:
- **רמה 0**: מדויקת, Circuit ארוכים יותר
- **רמה 1**: איזון טוב בין דיוק לקירוב, כריתת שערים עם זוויות ברבעון התחתון ה-10
- **רמה 2**: קירוב מעט גבוה יותר, כריתת שערים עם זוויות ברבעון התחתון ה-20 ושימוש ב-`approximation_degree=0.95` בתרגום
- **רמה 3**: רמת קירוב מקסימלית, כריתת שערים ברבעון התחתון ה-30 ושימוש ב-`approximation_degree=0.90` בתרגום

**רמות תרגום (0-5)**: שליטה בניסיונות אופטימיזציה מתקדמים של ה-Transpiler לקומפילציה של Circuit קוונטי. זה יכול להוביל לתקורה קלאסית מוגברת, ובמקרים מסוימים ייתכן שלא ישנה את עומק ה-Circuit. ערך ברירת המחדל `2` בדרך כלל מוביל ל-Circuit הקטן ביותר והוא יחסית מהיר.
- **רמה 0**: אופטימיזציה של Circuit DCQO המפורק (פריסה, ניתוב, תזמון)
- **רמה 1**: אופטימיזציה של `PauliEvolutionGate` ולאחר מכן Circuit DCQO המפורק (max_trials=10)
- **רמה 2**: אופטימיזציה של `PauliEvolutionGate` ולאחר מכן Circuit DCQO המפורק (max_trials=15)
- **רמה 3**: אופטימיזציה של `PauliEvolutionGate` ולאחר מכן Circuit DCQO המפורק (max_trials=20)
- **רמה 4**: אופטימיזציה של `PauliEvolutionGate` ולאחר מכן Circuit DCQO המפורק (max_trials=25)
- **רמה 5**: אופטימיזציה של `PauliEvolutionGate` ולאחר מכן Circuit DCQO המפורק (max_trials=50)

**רמות עיבוד לאחר (0-2)**: שליטה בכמות האופטימיזציה הקלאסית, המפצה על שגיאות היפוך סיבית עם מספרים שונים של מעברים חמדניים בחיפוש מקומי:
- **רמה 0**: מעבר אחד
- **רמה 1**: 2 מעברים
- **רמה 2**: 3 מעברים

**מצב Transpile-only**: זמין כעת למשתמשים שרוצים לנתח אופטימיזציה של Circuit ללא הרצת ביצוע האלגוריתם הקוונטי המלא.

**דוגמת תצורה מותאמת אישית**: כך ניתן להגדיר את Iskay עם הגדרות שונות:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**אופטימיזציית seed**: שים לב ש-`seed_transpiler` מוגדר ל-`None` כברירת מחדל. זה מאפשר את תהליך האופטימיזציה האוטומטי של ה-Transpiler. כאשר `None`, המערכת תתחיל ניסיון עם seeds מרובים ותבחר את זה המייצר את עומק ה-Circuit הטוב ביותר, תוך ניצול מלוא העוצמה של פרמטר `max_trials` עבור כל רמת תרגום.

**ביצועי רמת תרגום**: הגדלת מספר `max_trials` עם ערכים גבוהים יותר עבור `transpilation_level` תגדיל בהכרח את זמן התרגום, אך ייתכן שלא תמיד ישנה את ה-Circuit הסופי — זה תלוי מאוד במבנה ה-Circuit הספציפי ובמורכבותו. עבור חלק מה-Circuit/בעיות, עם זאת, ההבדל בין 10 ניסיונות (רמה 1) ל-50 ניסיונות (רמה 5) יכול להיות דרמטי, לכן חקירת פרמטרים אלה עשויה להיות המפתח למציאת פתרון בהצלחה.

### פלט
| שם   | סוג               | תיאור                                                           | דוגמה                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | פתרון ומטא-נתונים. המבנה משתנה בהתאם לאפשרות `transpile_only`. |   ראה "תוכן מילון התוצאות" להלן      |

### תוכן מילון התוצאות
מבנה מילון התוצאות תלוי במצב הביצוע:

| שדה         | סוג               | מצב | תיאור                                                                                                             | דוגמה |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **סטנדרטי** | הפתרון הממופה הממוין כאשר המפתחות הם אינדקסי משתנים (כמחרוזות) ממוינים מספרית והערכים הם ערכי המשתנים המתאימים (1/-1 לבעיות spin, 1/0 לבעיות בינאריות).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **סטנדרטי** | מידע מפורט על הפתרון (ראה פרטים להלן)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **סטנדרטי** | סוג בעיית האופטימיזציה ('spin' או 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Transpile-only** | ניתוח Circuit ופרטי תרגום (ראה פרטים להלן) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### ביצוע סטנדרטי
כאשר הפרמטר האופציונלי `transpile_only=False`:

**מילון solution_info:**
- **"bitstring"** (`str`): ייצוג המחרוזת הבינארית הגולמית של הפתרון.
- **"cost"** (`float`): ערך העלות/אנרגיה המשויך לפתרון.
- **"seed_transpiler"** (`int`): ה-seed האקראי בשימוש ב-Transpiler שייצר תוצאה זו.
- **"mapping"** (`Dict[int, int]`): מיפוי ה-Qubit-למשתנה המקורי בשימוש בחישוב.
- **"qpu_time"** (`float`, אופציונלי): זמן הביצוע של ה-QPU בשניות.

**הערות מיפוי משתנים:**
- מילון ה-`solution` מתקבל ממחרוזת הפתרון הבינארית, תוך שימוש באובייקט `mapping` לאינדוקס המשתנים.
- כאשר `problem_type=spin` אנו משתמשים בהשמה $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- מפתחות במילון הפתרון הם אינדקסי משתנים ממוינים מספרית כמחרוזות.

#### ניתוח תרגום
כאשר הפרמטר האופציונלי `transpile_only=True`:

**מילון transpilation_info:**
- **"best_seed"** (`int`): ה-seed האופטימלי שנמצא לתרגום
- **"transpilation_time_seconds"** (`float`): זמן שנדרש לתהליך התרגום
- **"transpiled_circuit"** (`Dict`): ניתוח Circuit המכיל:
  - **"depth"** (`int`): עומק ה-Circuit (מספר שכבות)
  - **"gate_count"** (`int`): מספר כולל של שערים ב-Circuit
  - **"num_qubits"** (`int`): מספר ה-Qubit בשימוש
  - **"width"** (`int`): רוחב ה-Circuit
  - **"operations"** (`Dict[str, int]`): ספירת כל סוג שער בשימוש

**שימוש במצב Transpile-only:**
- זמין למשתמשים שרוצים לנתח אופטימיזציה של Circuit ללא הרצת ביצוע האלגוריתם הקוונטי המלא.
- שימושי לניתוח Circuit, מחקרי אופטימיזציה של עומק, והבנת השפעות התרגום לפני מחויבות לביצוע מלא.
## התחלה
בתיעוד הזה נעבור על השלבים של שימוש ב-Iskay Quantum Optimizer. בתהליך נראה בקצרה איך לטעון את הפונקציה מהקטלוג ואיך להמיר את הבעיה שלך לקלט תקין, תוך כדי הדגמת אפשרויות הניסוי עם פרמטרים אופציונליים שונים.

לדוגמה מפורטת יותר, כדאי לבדוק את המדריך [פתרון בעיית פיצול השוק עם Iskay Quantum Optimizer של Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), שם עוברים על כל התהליך של שימוש ב-Iskay Solver לטיפול בבעיית פיצול השוק — אתגר הקצאת משאבים מהעולם האמיתי שבו שווקים צריכים להתחלק לאזורי מכירה מאוזנים כדי לעמוד ביעדי ביקוש מדויקים.

מתחברים עם מפתח ה-API שלך, שנמצא ב[לוח המחוונים של IBM Quantum Platform](http://quantum.cloud.ibm.com/), ובוחרים את Qiskit Function כך:

In [ ]:
# ruff: noqa: F821

> **Note:** הקוד הבא מניח שכבר שמרת את פרטי ההתחברות שלך. אם לא עשית זאת, עקוב אחר ההוראות ב[שמירת חשבון IBM Cloud שלך](/guides/functions#install-qiskit-functions-catalog-client) כדי להתאמת עם מפתח ה-API שלך.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## דוגמה 1: פונקציית עלות פשוטה

נשקול את פונקציית העלות בנוסחת ספין:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

כאשר $(x_0, ..., x_4) \in {-1, 1}^5$.

הפתרון לפונקציית העלות הפשוטה הזו הוא
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
עם ערך מינימלי $C^{*} = -6$

### 1. יצירת פונקציית המטרה

מתחילים ביצירת מילון עם המקדמים של פונקציית המטרה כך:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. הרצת המייעל
פותרים את הבעיה על ידי הרצת המייעל. מאחר ש-$(x_0, ..., x_4) \in {-1, 1}^5$ צריך להגדיר `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. שליפת התוצאה
הפתרון לבעיית האופטימיזציה מתקבל ישירות מהמייעל.

In [ ]:
print(job.result())

זה יציג מילון בצורה הבאה:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

שים לב שהמילון `solution` מציג את וקטור התוצאה $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## דוגמה 2: MaxCut

בעיות גרף רבות כמו MaxCut או Maximum independent set הן בעיות NP-hard ומועמדות אידיאליות לבדיקת אלגוריתמים קוונטיים וחומרה. הדוגמה הזו מדגימה פתרון בעיית MaxCut של גרף 3-רגולרי עם Quantum Optimizer.

כדי להריץ את הדוגמה הזו צריך להתקין את חבילת `networkx` בנוסף ל-`qiskit-ibm-catalog`. כדי להתקין אותה, מריצים את הפקודה הבאה:

In [ ]:
# %pip install networkx numpy

### 1. יצירת פונקציית המטרה
מתחילים ביצירת גרף 3-רגולרי אקראי. עבור הגרף הזה מגדירים את פונקציית המטרה של בעיית MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. הרצת המייעל
פותרים את הבעיה על ידי הרצת המייעל.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. שליפת התוצאה
שולפים את התוצאה וממפים את מחרוזת הביטים של הפתרון בחזרה לצמתים המקוריים של הגרף.

In [ ]:
print(job.result())

הפתרון לבעיית MaxCut כלול ישירות במילון המשנה `solution` של אובייקט התוצאה

In [ ]:
maxcut_solution = job.result()["solution"]

## דוגמה 3: מקרי בנצ'מארק
מקרי הבנצ'מארק זמינים ב-GitHub: [מקרי בנצ'מארק של Kipu.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

אפשר לטעון את המקרים באמצעות הספרייה `pygithub`. כדי להתקין אותה, הרץ את הפקודה הבאה:

In [ ]:
# %pip install pygithub

הנתיבים למקרי הבנצ'מארק הם:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

כדי לשחזר את ביצועי הבנצ'מארק עבור מקרי ה-HUBO, בחר את ה-Backend‏ `ibm_marrakesh` והגדר את `direct_qubit_mapping` לערך `True` בתת-מילון `options`.
הדוגמה הבאה מריצה את מקרה ה-Maxcut עם 32 צמתים.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## מקרי שימוש
מקרי שימוש טיפוסיים עבור פותר האופטימיזציה הם בעיות אופטימיזציה קומבינטורית. אפשר לפתור בעיות ממגוון תעשיות כמו פיננסים, תרופות או לוגיסטיקה. הנה כמה דוגמאות.
* אופטימיזציית תיק השקעות (QUBO): [פרסום מדעי](https://doi.org/10.1103/PhysRevApplied.22.054037) ו[נייר עמדה](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* קיפול חלבונים (HUBO): [פרסום מדעי](https://doi.org/10.1103/PhysRevApplied.20.014024)
* תזמון לוגיסטי (QUBO): [פרסום מדעי](https://doi.org/10.1103/PhysRevApplied.22.064068)
* אופטימיזציית רשתות: [וובינר](https://www.youtube.com/watch?v=w5SrCIK88No)
* פיצול שוק (QUBO): [מדריך](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

אם אתה מעוניין להתמודד עם מקרה שימוש ספציפי ולפתח מיפוי ייעודי, נוכל לסייע לך. [צור קשר.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
## קבלת תמיכה
לקבלת תמיכה, פנה אל support@kipu-quantum.com.
## צעדים הבאים
- [בקש גישה ל-Quantum Optimizer מאת Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- נסה את המדריך [פתרון בעיית פיצול השוק עם Iskay Quantum Optimizer של Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- עיין ב-[Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- עיין ב-[Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- עיין ב-[Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)
## מידע נוסף
Iskay, כמו שם החברה שלנו Kipu Quantum, הוא מילה פרואנית. למרות שאנחנו סטארטאפ מגרמניה, המילים האלו מגיעות ממולדתו של אחד ממייסדי-השותפים שלנו, שם ה-Quipu היה אחד ממכשירי החישוב הראשונים שפותחו על ידי האנושות לפני 2000 שנה לפני הספירה.